# 04 — Activations, Initialization, and Gradient Flow (Manual NumPy)


> **Learning contract.** Every code cell is preceded by an explanation of what the code does, why the operation exists mathematically, what tensor/array shapes are expected, and what production or business failure it prevents. Run the notebooks in numerical order in a fresh Conda environment.


Without a nonlinear activation, stacked dense layers collapse into a single affine transformation. ReLU creates piecewise-linear nonlinear behavior while avoiding the severe positive-side saturation of sigmoid/tanh. Initialization controls the distribution of pre-activations before training begins.

- **Xavier/Glorot** targets variance stability for approximately symmetric activations.
- **He/Kaiming** compensates for ReLU discarding roughly half of a symmetric signal.

Bad scale can saturate units, kill ReLUs or explode/vanish gradients.


## Code walkthrough — compare activation functions and their derivatives
The plots show both output and local derivative. Backpropagation multiplies upstream gradients by these derivatives, so a derivative near zero blocks learning even when the forward activation looks numerically valid.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
z = np.linspace(-6,6,400)
sig = 1/(1+np.exp(-z)); tanh=np.tanh(z); relu=np.maximum(z,0); leaky=np.where(z>0,z,0.05*z)
fig, axes=plt.subplots(1,2,figsize=(11,4))
for arr,label in [(sig,'sigmoid'),(tanh,'tanh'),(relu,'ReLU'),(leaky,'Leaky ReLU')]: axes[0].plot(z,arr,label=label)
for arr,label in [(sig*(1-sig),'sigmoid derivative'),(1-tanh**2,'tanh derivative'),((z>0).astype(float),'ReLU derivative'),(np.where(z>0,1,.05),'Leaky derivative')]: axes[1].plot(z,arr,label=label)
axes[0].legend(); axes[1].legend(); axes[0].set_title('activations'); axes[1].set_title('local derivatives'); plt.tight_layout(); plt.show()


## Code walkthrough — initialization scale changes signal propagation
We propagate a random batch through several random ReLU layers using three scales. The variance and percentage of zero activations expose why initialization is part of optimization, not mere bookkeeping.


In [ ]:
rng=np.random.default_rng(42); X=rng.normal(size=(1000,128))
for scale in [0.02, np.sqrt(2/128), 1.0]:
    A=X.copy(); stats=[]
    for depth in range(5):
        W=rng.normal(0,scale,size=(A.shape[1],128)); A=np.maximum(A@W,0); stats.append((A.std(),(A==0).mean()))
    print('scale',round(float(scale),4),'layer std/dead%',[(round(s,3),round(d,3)) for s,d in stats])


## Production implication
Initialization problems usually appear as training instability, NaNs, non-improving loss or highly variable runs. In a governed ML pipeline, seeds, initialization policy and optimizer configuration belong in experiment metadata.
